# FBSA — Sanity Checks

Cheap pre-flight checks before delegating long runs. ~5–10 min on A100. Should consume ≤1 Colab unit.

Pass criteria:
1. forward pass returns `[B, 1, 224, 224]`
2. trainable params in [1M, 5M] (encoder frozen)
3. 50-step overfit on 32 synthetic images: loss drops ≥30%, train Dice ≥0.7
4. one BRISC val pass returns finite metrics, HD in [10, 80] for the untrained model

In [ ]:
import sys
from pathlib import Path
PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))
%pip -q install kagglehub timm scipy

In [ ]:
import torch
from tumor_seg.config import TrainConfig
from tumor_seg.models import FBSASegmenter
from tumor_seg.metrics import dice_coefficient, iou_coefficient, hausdorff_distance_metric
from tumor_seg.losses import DiceBCELoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = TrainConfig()
model = FBSASegmenter().to(device)
print('device =', device)

In [ ]:
# Check 1: forward shape
with torch.no_grad():
    out = model(torch.randn(2, 3, 224, 224, device=device))
assert out.shape == (2, 1, 224, 224), out.shape
print('shape:', tuple(out.shape))

# Check 2: param freeze
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable={trainable/1e6:.2f}M / total={total/1e6:.2f}M')
assert 1e6 <= trainable <= 5e6

In [ ]:
# Check 3: mini-overfit on synthetic data
!python {PROJECT}/scripts/run_smoke.py

In [ ]:
# Check 4: untrained val pass on BRISC (run only if BRISC is mounted)
import kagglehub
data_root = Path(kagglehub.dataset_download('briscdataset/brisc2025')) / 'brisc2025' / 'segmentation_task'
from tumor_seg.data import create_brisc_dataloaders
_, val_loader = create_brisc_dataloaders(data_root, batch_size=8, image_size=224, val_limit=64)
model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    images, masks = images.to(device), masks.to(device)
    logits = model(images)
    print('dice =', dice_coefficient(logits, masks).item())
    print('iou  =', iou_coefficient(logits, masks).item())
    print('hd   =', hausdorff_distance_metric(logits, masks, image_size=224))